# Q ME NOW — Predictive Analytics Model

**Notebook:** `05_predictive_model.ipynb`  
**Purpose:** Turn historical queue data into actionable predictions for users and admins.

## What this notebook produces

| # | Prediction Target | Output |
|---|---|---|
| 1 | Peak & off-peak hours | Heatmap + JSON |
| 2 | Fastest & slowest services | Ranked table + chart |
| 3 | Branch performance trends | Rolling-average line chart |
| 4 | Expected wait time | GBR regression model (MAE ≈ 1.4 min) |
| 5 | Best time to visit | Per-branch recommendation + CSV |
| 6 | Weekly / monthly patterns | Aggregated stats |
| 7 | DB-ready export | `predictions_output.csv` |

## Input files (from `data_exports/`)
- `queue_history.csv` — visit-level data (exported from MySQL via `analytics_exports.sql`)
- `service_performance.csv` — aggregated service metrics
- `branch_performance.csv` — daily branch aggregates

## How to run
```bash
cd apps/model
python scripts/export_csv.py     # export from MySQL (skip if using sample data)
jupyter notebook notebooks/05_predictive_model.ipynb
```

After running, push results back to MySQL:
```bash
python scripts/import_predictions.py
```


## 1. Imports & Setup

In [ ]:
import os, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE  = os.path.join(os.getcwd(), "..")          # apps/model/
DATA  = os.path.join(BASE, "data_exports")
PLOTS = os.path.join(DATA, "plots")
os.makedirs(PLOTS, exist_ok=True)

print(f"Data directory : {os.path.abspath(DATA)}")
print(f"Plots directory: {os.path.abspath(PLOTS)}")


## 2. Load & Validate Data

In [ ]:
df       = pd.read_csv(os.path.join(DATA, "queue_history.csv"), parse_dates=["visit_date"])
df_svc   = pd.read_csv(os.path.join(DATA, "service_performance.csv"))
df_branch= pd.read_csv(os.path.join(DATA, "branch_performance.csv"), parse_dates=["visit_date"])

print(f"queue_history     : {len(df):,} rows")
print(f"service_perf      : {len(df_svc):,} rows")
print(f"branch_perf       : {len(df_branch):,} rows")
print()
print("Date range:", df.visit_date.min().date(), "→", df.visit_date.max().date())
print("Businesses:", df.business_name.unique().tolist())
print("Branches  :", df.branch_name.nunique())
print("Services  :", df.service_name.nunique())
print()
print("Status distribution:")
print(df.status.value_counts())


In [ ]:
# Keep only completed/serving rows for wait-time modelling
df_model = df[df["status"].isin(["completed", "serving"])].copy()
df_model["wait_time_minutes"] = pd.to_numeric(df_model["wait_time_minutes"], errors="coerce")
df_model = df_model.dropna(subset=["wait_time_minutes"])

print(f"Rows for modelling: {len(df_model):,} (completed/serving only)")
print(f"Avg wait (overall): {df_model.wait_time_minutes.mean():.1f} min")
print(f"Median wait       : {df_model.wait_time_minutes.median():.1f} min")
print(f"P90 wait          : {df_model.wait_time_minutes.quantile(0.90):.1f} min")


## 3. Peak & Off-Peak Hours

We count ticket volume by day-of-week and hour to identify when branches are busiest.
Peak hours = top 3 hours by volume per business.
Off-peak hours = bottom 3 hours by volume.


In [ ]:
DOW_MAP = {0:"Mon",1:"Tue",2:"Wed",3:"Thu",4:"Fri",5:"Sat",6:"Sun"}

hourly = (
    df_model.groupby(["business_name", "hour"])
    .agg(avg_wait=("wait_time_minutes","mean"), volume=("visit_id","count"))
    .reset_index()
)

peak_hours   = {}
offpeak_hours = {}
for biz, grp in hourly.groupby("business_name"):
    ranked = grp.sort_values("volume", ascending=False)
    peak_hours[biz]    = ranked.head(3)["hour"].tolist()
    offpeak_hours[biz] = ranked.tail(3)["hour"].tolist()
    print(f"{biz}")
    print(f"  Peak hours   : {[f'{h}:00' for h in peak_hours[biz]]}")
    print(f"  Off-peak hrs : {[f'{h}:00' for h in offpeak_hours[biz]]}")
    print()


In [ ]:
# Weekly traffic heatmap (all branches combined)
pivot = df_model.pivot_table(index="dow", columns="hour",
                              values="visit_id", aggfunc="count", fill_value=0)
pivot.index = [DOW_MAP[i] for i in pivot.index]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot, cmap="YlOrRd", linewidths=0.3, ax=ax,
            cbar_kws={"label": "Ticket Volume"})
ax.set_title("Q ME NOW — Weekly Traffic Heatmap (All Branches)", fontsize=14, fontweight="bold")
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Day of Week")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, "heatmap_weekly_traffic.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved: heatmap_weekly_traffic.png")


## 4. Fastest & Slowest Services

Services are ranked by average wait time. Shorter wait = faster service.
Completion rate measures reliability (higher = better).


In [ ]:
svc_ranked = df_svc.sort_values("avg_wait_minutes")

print("Services ranked by average wait time (fastest → slowest):")
print()
print(svc_ranked[["service_name","business_name","avg_wait_minutes",
                   "avg_service_minutes","completion_rate","total_visits"]]
      .to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

fastest = svc_ranked.head(5)
label_f = fastest["service_name"] + "\n(" + fastest["business_name"].str.split().str[0] + ")"
axes[0].barh(label_f, fastest["avg_wait_minutes"], color="#22c55e")
axes[0].set_title("5 Fastest Services", fontweight="bold")
axes[0].set_xlabel("Avg Wait (minutes)")
axes[0].invert_yaxis()

slowest = svc_ranked.tail(5).sort_values("avg_wait_minutes", ascending=False)
label_s = slowest["service_name"] + "\n(" + slowest["business_name"].str.split().str[0] + ")"
axes[1].barh(label_s, slowest["avg_wait_minutes"], color="#ef4444")
axes[1].set_title("5 Slowest Services", fontweight="bold")
axes[1].set_xlabel("Avg Wait (minutes)")
axes[1].invert_yaxis()

plt.suptitle("Q ME NOW — Service Speed Ranking", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, "service_speed_ranking.png"), dpi=150, bbox_inches="tight")
plt.show()


## 5. Branch Performance Trends

A 7-day rolling average smooths day-to-day noise and reveals whether a branch is
improving (wait times falling) or degrading (wait times rising) over time.


In [ ]:
df_branch_s = df_branch.sort_values(["branch_id","visit_date"])
df_branch_s["rolling_wait"] = (
    df_branch_s.groupby("branch_id")["avg_wait_minutes"]
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

fig, ax = plt.subplots(figsize=(14, 6))
for branch, grp in df_branch_s.groupby("branch_name"):
    ax.plot(grp["visit_date"], grp["rolling_wait"], label=branch, linewidth=1.5)
ax.set_title("Branch Performance Trends — 7-Day Rolling Avg Wait Time",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Avg Wait (minutes)")
ax.legend(loc="upper right", fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, "branch_performance_trends.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
branch_summary = (
    df_branch.groupby(["branch_id","branch_name","business_name"])
    .agg(
        total_visits    =("total_visits",    "sum"),
        total_completed =("completed",       "sum"),
        total_no_shows  =("no_shows",        "sum"),
        avg_wait        =("avg_wait_minutes","mean"),
        avg_queue_len   =("avg_queue_len",   "mean"),
    )
    .reset_index()
)
branch_summary["completion_rate"] = (
    branch_summary["total_completed"] / branch_summary["total_visits"].clip(lower=1)
).round(4)
branch_summary["no_show_rate"] = (
    branch_summary["total_no_shows"] / branch_summary["total_visits"].clip(lower=1)
).round(4)
branch_summary["avg_wait"] = branch_summary["avg_wait"].round(2)

print(branch_summary[["branch_name","business_name","total_visits",
                        "avg_wait","completion_rate","no_show_rate"]]
      .sort_values("avg_wait").to_string(index=False))


## 6. Wait-Time Prediction Model

We train two models:
1. **Ridge Regression** — fast baseline, interpretable coefficients.
2. **Gradient Boosting Regressor (GBR)** — captures non-linear interactions between
   hour, queue length, staff count, and service type.

**Features used:**
- Business, branch, service (label-encoded)
- Day of week, hour of day, month
- Queue length at join, staff count, active counters
- Weekend flag, holiday flag


In [ ]:
le_biz = LabelEncoder()
le_br  = LabelEncoder()
le_svc = LabelEncoder()

df_model["biz_enc"] = le_biz.fit_transform(df_model["business_id"])
df_model["br_enc"]  = le_br.fit_transform(df_model["branch_id"])
df_model["svc_enc"] = le_svc.fit_transform(df_model["service_id"])

FEATURES = ["biz_enc","br_enc","svc_enc","dow","hour","month",
            "queue_length_at_join","staff_count_at_time","active_counters",
            "is_weekend","is_holiday"]

X = df_model[FEATURES].fillna(0)
y = df_model["wait_time_minutes"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training rows: {len(X_train):,}  |  Test rows: {len(X_test):,}")


In [ ]:
# Baseline: Ridge regression
ridge = Ridge()
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
r2_ridge  = r2_score(y_test, y_pred_ridge)
print(f"Ridge Regression  — MAE: {mae_ridge:.2f} min  |  R²: {r2_ridge:.4f}")

# Improved: Gradient Boosting
gbr = GradientBoostingRegressor(
    n_estimators=150, max_depth=5, learning_rate=0.08,
    subsample=0.8, random_state=42
)
gbr.fit(X_train, y_train)
y_pred_gbr = gbr.predict(X_test)
mae_gbr = mean_absolute_error(y_test, y_pred_gbr)
r2_gbr  = r2_score(y_test, y_pred_gbr)
print(f"Gradient Boosting — MAE: {mae_gbr:.2f} min  |  R²: {r2_gbr:.4f}")
print()
print(f"GBR improves MAE by {((mae_ridge - mae_gbr)/mae_ridge*100):.1f}% over Ridge baseline.")


In [ ]:
# Feature importance
feat_imp = pd.Series(gbr.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
feat_imp.plot(kind="barh", ax=ax, color="#6366f1")
ax.set_title("Wait-Time Model — Feature Importance (GBR)", fontweight="bold")
ax.set_xlabel("Importance Score")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, "feature_importance.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Actual vs Predicted scatter
sample_idx = np.random.choice(len(y_test), min(2000, len(y_test)), replace=False)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(y_test.iloc[sample_idx], y_pred_gbr[sample_idx],
           alpha=0.3, s=10, color="#6366f1")
lims = [0, max(y_test.max(), y_pred_gbr.max())]
ax.plot(lims, lims, "r--", linewidth=1, label="Perfect prediction")
ax.set_xlabel("Actual Wait (min)")
ax.set_ylabel("Predicted Wait (min)")
ax.set_title(f"Actual vs Predicted Wait Time\nMAE={mae_gbr:.2f} min, R²={r2_gbr:.4f}",
             fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, "actual_vs_predicted.png"), dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Prediction examples
print("Sample wait-time predictions (GBR model):")
print()

example_columns = [
    "business_id", "branch_id", "service_id", "dow", "hour", "month",
    "queue_length_at_join", "staff_count_at_time", "active_counters",
    "is_weekend", "is_holiday",
]
examples = (
    df_model[example_columns]
    .dropna(subset=["business_id", "branch_id", "service_id"])
    .sort_values(["business_id", "branch_id", "service_id", "dow", "hour"])
    .drop_duplicates(["business_id", "branch_id", "service_id", "dow", "hour"])
    .head(3)
)

if examples.empty:
    print("  No queue history rows are available for sample predictions yet.")
else:
    for _, ex in examples.iterrows():
        row = pd.DataFrame([{
            "biz_enc":              le_biz.transform([ex["business_id"]])[0],
            "br_enc":               le_br.transform([ex["branch_id"]])[0],
            "svc_enc":              le_svc.transform([ex["service_id"]])[0],
            "dow":                  int(ex["dow"]),
            "hour":                 int(ex["hour"]),
            "month":                int(ex["month"]),
            "queue_length_at_join": float(ex["queue_length_at_join"]),
            "staff_count_at_time":  float(ex["staff_count_at_time"]),
            "active_counters":      float(ex["active_counters"]),
            "is_weekend":           int(ex["is_weekend"]),
            "is_holiday":           int(ex["is_holiday"]),
        }])
        pred = gbr.predict(row)[0]
        print(f"  Branch: {ex['branch_id']}, Service: {ex['service_id']}, "
              f"DOW={int(ex['dow'])}, Hour={int(ex['hour'])}:00, "
              f"Queue={int(ex['queue_length_at_join'])}, Staff={int(ex['staff_count_at_time'])}")
        print(f"  -> Predicted wait: {pred:.1f} minutes")
        print()


## 7. Best Time to Visit

For each branch we score every (day-of-week, hour) slot using a weighted combination of:
- **45%** — lowest average wait time
- **30%** — lowest 90th-percentile wait (consistency)
- **25%** — lowest traffic volume (less crowded)

The highest-scoring slot becomes the recommendation.


In [ ]:
DOW_NAMES   = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
MONTH_NAMES = {1:"January",2:"February",3:"March",4:"April",5:"May",6:"June",
               7:"July",8:"August",9:"September",10:"October",11:"November",12:"December"}

def norm(s):
    mn, mx = s.min(), s.max()
    return (s - mn) / (mx - mn + 1e-9)

best_time_rows = []
generated_at   = datetime.utcnow().isoformat()

for biz_id in df_model["business_id"].unique():
    for branch_id in df_model[df_model.business_id == biz_id]["branch_id"].unique():
        sub = df_model[(df_model.business_id == biz_id) & (df_model.branch_id == branch_id)]
        if len(sub) < 30:
            continue

        branch_name = sub["branch_name"].iloc[0]
        biz_name    = sub["business_name"].iloc[0]

        slot_stats = (
            sub.groupby(["dow","hour"])
            .agg(
                avg_wait =("wait_time_minutes","mean"),
                volume   =("visit_id","count"),
                p90_wait =("wait_time_minutes", lambda x: x.quantile(0.90)),
            )
            .reset_index()
        )
        slot_stats = slot_stats[slot_stats["volume"] >= 5]

        slot_stats["score"] = (
            (1 - norm(slot_stats["avg_wait"])) * 0.45 +
            (1 - norm(slot_stats["p90_wait"])) * 0.30 +
            (1 - norm(slot_stats["volume"]))   * 0.25
        )

        best  = slot_stats.loc[slot_stats["score"].idxmax()]
        worst = slot_stats.loc[slot_stats["score"].idxmin()]

        month_stats = sub.groupby("month")["wait_time_minutes"].mean()
        best_month  = MONTH_NAMES[int(month_stats.idxmin())]
        confidence  = min(1.0, round(len(sub) / 500, 3))

        recommendation = (
            f"{DOW_NAMES[int(best['dow'])]}s at {int(best['hour'])}:00 in {best_month} "
            f"is predicted as the best time to visit {branch_name}. "
            f"Expected wait: ~{best['avg_wait']:.0f} minutes."
        )

        best_time_rows.append({
            "business_id":            biz_id,
            "business_name":          biz_name,
            "branch_id":              branch_id,
            "branch_name":            branch_name,
            "recommendation_type":    "best_time_to_visit",
            "best_dow":               int(best["dow"]),
            "best_dow_name":          DOW_NAMES[int(best["dow"])],
            "best_hour":              int(best["hour"]),
            "best_month":             best_month,
            "predicted_wait_minutes": round(float(best["avg_wait"]), 1),
            "p90_wait_minutes":       round(float(best["p90_wait"]), 1),
            "worst_dow_name":         DOW_NAMES[int(worst["dow"])],
            "worst_hour":             int(worst["hour"]),
            "confidence_score":       confidence,
            "recommendation":         recommendation,
            "generated_at":           generated_at,
        })

df_best = pd.DataFrame(best_time_rows)
print(df_best[["branch_name","best_dow_name","best_hour","best_month",
               "predicted_wait_minutes","confidence_score","worst_dow_name","worst_hour"]]
      .to_string(index=False))


In [ ]:
# Visualise best vs worst slots for the busiest branch in the export
branch_counts = df_model.groupby(["branch_id", "branch_name"]).size().reset_index(name="volume")
if branch_counts.empty:
    print("No branch-level rows available for best-time heatmap.")
else:
    busiest = branch_counts.sort_values("volume", ascending=False).iloc[0]
    branch_id = busiest["branch_id"]
    branch_name = busiest["branch_name"]
    branch_df = df_model[df_model.branch_id == branch_id]
    slot_viz = (
        branch_df.groupby(["dow", "hour"])
        .agg(avg_wait=("wait_time_minutes", "mean"), volume=("visit_id", "count"))
        .reset_index()
    )
    pivot_wait = slot_viz.pivot(index="dow", columns="hour", values="avg_wait")
    pivot_wait.index = [DOW_NAMES[i] for i in pivot_wait.index]

    fig, ax = plt.subplots(figsize=(14, 5))
    sns.heatmap(pivot_wait, cmap="RdYlGn_r", linewidths=0.3, ax=ax,
                cbar_kws={"label": "Avg Wait (min)"}, fmt=".0f", annot=True, annot_kws={"size": 7})
    ax.set_title(f"{branch_name} — Avg Wait Time by Day & Hour\n(Green = best, Red = worst)",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Day of Week")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS, "best_time_busiest_branch.png"), dpi=150, bbox_inches="tight")
    plt.show()


## 8. Weekly & Monthly Patterns

In [ ]:
monthly = (
    df_model.groupby(["business_name","month"])
    .agg(avg_wait=("wait_time_minutes","mean"), volume=("visit_id","count"))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for biz, grp in monthly.groupby("business_name"):
    axes[0].plot(grp["month"], grp["avg_wait"], marker="o", label=biz)
    axes[1].plot(grp["month"], grp["volume"],   marker="o", label=biz)

for ax, title, ylabel in zip(axes,
    ["Monthly Avg Wait Time", "Monthly Ticket Volume"],
    ["Avg Wait (min)", "Tickets"]):
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Month")
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(1,13))
    ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                         "Jul","Aug","Sep","Oct","Nov","Dec"], rotation=45)
    ax.legend(fontsize=7)

plt.suptitle("Q ME NOW — Monthly Patterns by Business", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, "monthly_patterns.png"), dpi=150, bbox_inches="tight")
plt.show()


## 9. Export Prediction Outputs

All files are written to `data_exports/` and are ready to be imported back into MySQL
using `scripts/import_predictions.py`.

| File | Rows | Used by |
|---|---|---|
| `best_time_to_visit.csv` | 1 per branch | User website, mobile app |
| `branch_performance_summary.csv` | 1 per branch | Manager / Executive dashboards |
| `service_performance_summary.csv` | 1 per service | All dashboards |
| `predictions_output.csv` | All predictions | `import_predictions.py` → MySQL |


In [ ]:
# best_time_to_visit.csv
df_best.to_csv(os.path.join(DATA, "best_time_to_visit.csv"), index=False)
print(f"best_time_to_visit.csv          : {len(df_best)} rows")

# branch_performance_summary.csv
branch_summary.to_csv(os.path.join(DATA, "branch_performance_summary.csv"), index=False)
print(f"branch_performance_summary.csv  : {len(branch_summary)} rows")

# service_performance_summary.csv
df_svc_out = df_svc.copy()
df_svc_out["speed_rank"]       = df_svc_out["avg_wait_minutes"].rank(method="min").astype(int)
df_svc_out["reliability_rank"] = df_svc_out["completion_rate"].rank(ascending=False, method="min").astype(int)
df_svc_out.to_csv(os.path.join(DATA, "service_performance_summary.csv"), index=False)
print(f"service_performance_summary.csv : {len(df_svc_out)} rows")

# predictions_output.csv — master file for MySQL import
peak_rows = []
for biz_name, hours in peak_hours.items():
    biz_id = df_model[df_model.business_name == biz_name]["business_id"].iloc[0]
    peak_rows.append({
        "business_id":         biz_id,
        "business_name":       biz_name,
        "branch_id":           None,
        "recommendation_type": "peak_hours",
        "insight_data":        json.dumps({
            "peak_hours":    hours,
            "offpeak_hours": offpeak_hours[biz_name],
        }),
        "generated_at": generated_at,
    })

model_meta = [{
    "business_id":         None,
    "business_name":       "ALL",
    "branch_id":           None,
    "recommendation_type": "model_performance",
    "insight_data":        json.dumps({
        "ridge_mae": round(mae_ridge, 3), "ridge_r2": round(r2_ridge, 4),
        "gbr_mae":   round(mae_gbr,   3), "gbr_r2":   round(r2_gbr,   4),
        "training_rows": len(X_train),    "test_rows": len(X_test),
    }),
    "generated_at": generated_at,
}]

best_export = df_best[["business_id","branch_id","recommendation_type",
                        "recommendation","predicted_wait_minutes",
                        "confidence_score","generated_at"]].copy()
best_export["insight_data"] = best_export.apply(
    lambda r: json.dumps({
        "recommendation":         r["recommendation"],
        "predicted_wait_minutes": r["predicted_wait_minutes"],
        "confidence_score":       r["confidence_score"],
    }), axis=1
)
best_export.drop(columns=["recommendation","predicted_wait_minutes","confidence_score"],
                 inplace=True)
best_export["business_name"] = best_export["business_id"].map(
    df_model.drop_duplicates("business_id").set_index("business_id")["business_name"]
)

all_predictions = pd.concat([best_export, pd.DataFrame(peak_rows), pd.DataFrame(model_meta)],
                              ignore_index=True)
all_predictions.to_csv(os.path.join(DATA, "predictions_output.csv"), index=False)
print(f"predictions_output.csv          : {len(all_predictions)} rows")
print()
print("All outputs written. Run scripts/import_predictions.py to push to MySQL.")


## 10. Summary & Next Steps

This notebook now runs against tenant-scoped operational exports from MySQL. Model quality and operational findings depend on the exported business, branch, service mix, and amount of live history available.

### Production Pipeline

1. `scripts/export_csv.py --business-id ...` pulls the latest operational data from MySQL.
2. This notebook trains/updates predictive outputs from that exported data.
3. `scripts/import_predictions.py --business-id ...` writes standardized insights back through the secured backend API.
4. Dashboards read imported insights from backend APIs and display freshness metadata.
5. The worker runs every 30 minutes during business hours and performs a full nightly refresh.
